# LeetCode 1646 — Get Maximum in Generated Array bằng Python

Link đề: https://leetcode.com/problems/get-maximum-in-generated-array/

Độ khó: **Easy** · Chủ đề: Array, Simulation

Skeleton LeetCode:

```python
class Solution:
    def getMaximumGenerated(self, n: int) -> int:

```

Mục tiêu học:

1. Đọc hiểu một **quy tắc sinh mảng** và chuyển nó thành code mô phỏng (simulation).
2. Phân biệt hai cách viết vòng lặp: theo `i` (ghi ra `2i`, `2i+1`) và theo chỉ số đích `k` (đọc từ `k // 2`).
3. Nắm chắc `//`, `%`, chẵn/lẻ và điều kiện biên `n = 0`, `n = 1`.
4. Làm quen với **đệ quy + ghi nhớ** (`functools.lru_cache`) như một cách nhìn khác.
5. Biết khi nào có thể **tính sẵn** (precompute) vì constraints nhỏ.
6. Tự viết lại lời giải và giải các biến thể dãy sinh theo quy tắc.

## 1. Bài toán yêu cầu gì?

Cho số nguyên `n`. Tạo mảng `nums` dài `n + 1` (chỉ số `0..n`) theo quy tắc:

- `nums[0] = 0`
- `nums[1] = 1`
- `nums[2 * i] = nums[i]` khi `2 <= 2 * i <= n`
- `nums[2 * i + 1] = nums[i] + nums[i + 1]` khi `2 <= 2 * i + 1 <= n`

Trả về **giá trị lớn nhất** trong `nums`.

| n | nums | Kết quả |
|---:|---|---:|
| `7` | `[0,1,1,2,1,3,2,3]` | `3` |
| `2` | `[0,1,1]` | `1` |
| `3` | `[0,1,1,2]` | `2` |
| `0` | `[0]` | `0` |
| `1` | `[0,1]` | `1` |

Constraints: `0 <= n <= 100`.

Chú ý hint của đề: **"Make sure not to fall in the base case of 0"** — khi `n = 0`, mảng chỉ có `nums[0]`, không được gán `nums[1]`.

### Tính tay với n = 7

| chỉ số | công thức | giá trị |
|---:|---|---:|
| 0 | gốc | 0 |
| 1 | gốc | 1 |
| 2 | `2·1` → `nums[1]` | 1 |
| 3 | `2·1+1` → `nums[1] + nums[2]` | 2 |
| 4 | `2·2` → `nums[2]` | 1 |
| 5 | `2·2+1` → `nums[2] + nums[3]` | 3 |
| 6 | `2·3` → `nums[3]` | 2 |
| 7 | `2·3+1` → `nums[3] + nums[4]` | 3 |

Max = `3`. Để ý: mọi phần tử chỉ phụ thuộc vào phần tử **có chỉ số nhỏ hơn** → tính từ trái sang phải là được.

In [1]:
nums = [0, 1, 1, 2, 1, 3, 2, 3]
print("nums =", nums)
print("max  =", max(nums))
print("độ dài =", len(nums), "= n + 1 với n =", len(nums) - 1)

nums = [0, 1, 1, 2, 1, 3, 2, 3]
max  = 3
độ dài = 8 = n + 1 với n = 7


## 2. Nền tảng Python cần biết

### 2.1. Tạo list có sẵn độ dài: `[0] * (n + 1)`

Ta cần mảng dài `n + 1` để gán theo chỉ số. `[0] * (n + 1)` tạo list gồm `n + 1` số 0.

Gán `nums[k] = ...` chỉ hợp lệ khi `0 <= k < len(nums)`; ngoài khoảng sẽ báo `IndexError`.

In [2]:
n = 4
nums = [0] * (n + 1)
print(nums, "độ dài", len(nums))
nums[4] = 9
print(nums)
try:
    nums[5] = 1
except IndexError as e:
    print("IndexError:", e)

[0, 0, 0, 0, 0] độ dài 5
[0, 0, 0, 0, 9]
IndexError: list assignment index out of range


### 2.2. Chia nguyên `//`, chia dư `%`, chẵn và lẻ

- `k // 2`: phần nguyên của `k / 2`.
- `k % 2`: số dư khi chia cho 2 — `0` là chẵn, `1` là lẻ.

Nếu `k = 2i` thì `k // 2 = i`. Nếu `k = 2i + 1` thì `k // 2 = i` vẫn thế! Đây là chìa khóa để viết vòng lặp theo `k`.

In [3]:
for k in range(2, 9):
    loai = "chẵn" if k % 2 == 0 else "lẻ"
    print(f"k={k}: k//2={k // 2}, k%2={k % 2} ({loai})")

k=2: k//2=1, k%2=0 (chẵn)
k=3: k//2=1, k%2=1 (lẻ)
k=4: k//2=2, k%2=0 (chẵn)
k=5: k//2=2, k%2=1 (lẻ)
k=6: k//2=3, k%2=0 (chẵn)
k=7: k//2=3, k%2=1 (lẻ)
k=8: k//2=4, k%2=0 (chẵn)


### 2.3. `range(start, stop)` và `range` có bước

- `range(2, n + 1)`: các số `2, 3, ..., n` (không gồm `n + 1`).
- `range(1, n // 2 + 1)`: các `i` sao cho `2i <= n`.

In [4]:
n = 7
print(list(range(2, n + 1)))
print(list(range(1, n // 2 + 1)))
print(list(range(0, 10, 3)), "(bước 3)")

[2, 3, 4, 5, 6, 7]
[1, 2, 3]
[0, 3, 6, 9] (bước 3)


### 2.4. `max()` và trường hợp `n` nhỏ

`max(list)` trả phần tử lớn nhất. List rỗng thì `max` báo lỗi — nhưng mảng của ta luôn dài ít nhất 1 (`n >= 0`).

In [5]:
print(max([0]))
print(max([0, 1]))
print(max([0, 1, 1, 2, 1, 3, 2, 3]))

0
1
3


## 3. Cách trực giác — mô phỏng đúng theo `i` trong đề

Đề viết quy tắc theo `i`: với mỗi `i`, ghi `nums[2i]` và `nums[2i+1]`. Ta dịch thẳng:

```text
for i = 1, 2, 3, ...:
    nếu 2i <= n:      nums[2i]   = nums[i]
    nếu 2i + 1 <= n:  nums[2i+1] = nums[i] + nums[i+1]
```

Bắt đầu `i = 1` vì điều kiện `2 <= 2i` ⇔ `i >= 1`.

Câu hỏi: khi tính `nums[2i+1]` ta cần `nums[i+1]` — nó đã có chưa? `i + 1 <= 2i` khi `i >= 1`, và với `i = 1` thì `nums[2] = nums[1]` vừa gán ngay trước đó. Với `i >= 2`, `i + 1 < 2i` nên đã được tính ở vòng trước. An toàn.

In [6]:
def get_max_by_i(n: int) -> int:
    if n == 0:
        return 0
    nums = [0] * (n + 1)
    nums[1] = 1
    i = 1
    while 2 * i <= n:
        nums[2 * i] = nums[i]
        if 2 * i + 1 <= n:
            nums[2 * i + 1] = nums[i] + nums[i + 1]
        i += 1
    return max(nums)

for n in [7, 2, 3, 0, 1]:
    print(f"n={n}: {get_max_by_i(n)}")

n=7: 3
n=2: 1
n=3: 2
n=0: 0
n=1: 1


### Nút thắt của cách này

Không phải nút thắt về tốc độ (vẫn `O(n)`), mà là về **độ dễ sai**:

- Phải kiểm tra hai điều kiện `2i <= n` và `2i + 1 <= n` riêng.
- Phải chứng minh `nums[i+1]` đã có trước khi dùng.
- Dễ quên `n == 0` (gán `nums[1]` sẽ lỗi vì mảng chỉ dài 1).

Có cách viết **gọn hơn và an toàn hơn**: duyệt theo **chỉ số đích** `k`.

In [7]:
def buggy_no_base_case(n):
    nums = [0] * (n + 1)
    nums[1] = 1           # lỗi khi n = 0
    return max(nums)

try:
    buggy_no_base_case(0)
except IndexError as e:
    print("n = 0 gây lỗi:", e)

n = 0 gây lỗi: list assignment index out of range


## 4. Insight — viết theo chỉ số đích `k`

Với mỗi `k` từ `2` đến `n`:

- `k` chẵn: `k = 2i` với `i = k // 2` → `nums[k] = nums[k // 2]`.
- `k` lẻ: `k = 2i + 1` với `i = k // 2` → `nums[k] = nums[k // 2] + nums[k // 2 + 1]`.

Vì `k // 2 < k` và `k // 2 + 1 < k` (khi `k >= 3`), các giá trị cần dùng **đã được tính** khi đi từ trái sang phải.

Đây là dạng **quy hoạch động** (dynamic programming) đơn giản: mỗi ô được tính từ các ô trước.

In [8]:
for k in range(2, 8):
    half = k // 2
    if k % 2 == 0:
        print(f"k={k} chẵn: nums[{k}] = nums[{half}]")
    else:
        print(f"k={k} lẻ  : nums[{k}] = nums[{half}] + nums[{half + 1}]")

k=2 chẵn: nums[2] = nums[1]
k=3 lẻ  : nums[3] = nums[1] + nums[2]
k=4 chẵn: nums[4] = nums[2]
k=5 lẻ  : nums[5] = nums[2] + nums[3]
k=6 chẵn: nums[6] = nums[3]
k=7 lẻ  : nums[7] = nums[3] + nums[4]


## 5. Lời giải chính — duyệt `k` từ 2 đến n

Giảng từng đoạn:

**Đoạn 1 — base case:** `n < 2` thì đáp án chính là `n` (mảng `[0]` hoặc `[0, 1]`).

**Đoạn 2 — tạo mảng và gán gốc:** `nums = [0] * (n + 1)`, `nums[1] = 1`.

**Đoạn 3 — điền từ trái sang phải theo chẵn/lẻ.**

**Đoạn 4 — trả `max(nums)`.**

In [9]:
def get_max_generated(n: int) -> int:
    if n < 2:
        return n
    nums = [0] * (n + 1)
    nums[1] = 1
    for k in range(2, n + 1):
        if k % 2 == 0:
            nums[k] = nums[k // 2]
        else:
            nums[k] = nums[k // 2] + nums[k // 2 + 1]
    return max(nums)

for n in [7, 2, 3, 0, 1, 100]:
    print(f"n={n}: {get_max_generated(n)}")

n=7: 3
n=2: 1
n=3: 2
n=0: 0
n=1: 1
n=100: 21


## 6. TRACE từng bước

Ta in ra từng `k`: chẵn hay lẻ, công thức đọc từ đâu, giá trị mới, mảng sau khi điền và max hiện tại.

Bảng trace (n = 7):

| k | loại | đọc | nums[k] | max đến k |
|---:|---|---|---:|---:|
| 2 | chẵn | nums[1]=1 | 1 | 1 |
| 3 | lẻ | nums[1]+nums[2]=1+1 | 2 | 2 |
| 4 | chẵn | nums[2]=1 | 1 | 2 |
| 5 | lẻ | nums[2]+nums[3]=1+2 | 3 | 3 |
| 6 | chẵn | nums[3]=2 | 2 | 3 |
| 7 | lẻ | nums[3]+nums[4]=2+1 | 3 | 3 |

In [10]:
def trace_generated(n: int) -> int:
    if n < 2:
        print(f"n={n} < 2 -> trả về {n}")
        return n
    nums = [0] * (n + 1)
    nums[1] = 1
    best = 1
    print("khởi tạo:", nums)
    for k in range(2, n + 1):
        h = k // 2
        if k % 2 == 0:
            nums[k] = nums[h]
            how = f"chẵn: nums[{h}] = {nums[h]}"
        else:
            nums[k] = nums[h] + nums[h + 1]
            how = f"lẻ  : nums[{h}] + nums[{h+1}] = {nums[h]} + {nums[h+1]}"
        best = max(best, nums[k])
        print(f"k={k} {how:<32} -> nums[{k}]={nums[k]} | max={best}")
    print("mảng cuối:", nums)
    return best

trace_generated(7)

khởi tạo: [0, 1, 0, 0, 0, 0, 0, 0]
k=2 chẵn: nums[1] = 1                -> nums[2]=1 | max=1
k=3 lẻ  : nums[1] + nums[2] = 1 + 1  -> nums[3]=2 | max=2
k=4 chẵn: nums[2] = 1                -> nums[4]=1 | max=2
k=5 lẻ  : nums[2] + nums[3] = 1 + 2  -> nums[5]=3 | max=3
k=6 chẵn: nums[3] = 2                -> nums[6]=2 | max=3
k=7 lẻ  : nums[3] + nums[4] = 2 + 1  -> nums[7]=3 | max=3
mảng cuối: [0, 1, 1, 2, 1, 3, 2, 3]


3

In [11]:
trace_generated(0)
trace_generated(1)

n=0 < 2 -> trả về 0
n=1 < 2 -> trả về 1


1

## 7. Code nộp LeetCode

In [12]:
class Solution:
    def getMaximumGenerated(self, n: int) -> int:
        if n < 2:
            return n
        nums = [0] * (n + 1)
        nums[1] = 1
        for k in range(2, n + 1):
            if k % 2 == 0:
                nums[k] = nums[k // 2]
            else:
                nums[k] = nums[k // 2] + nums[k // 2 + 1]
        return max(nums)

sol = Solution()
assert sol.getMaximumGenerated(7) == 3
assert sol.getMaximumGenerated(2) == 1
assert sol.getMaximumGenerated(3) == 2
assert sol.getMaximumGenerated(0) == 0
assert sol.getMaximumGenerated(1) == 1
print("Ví dụ đề + base case PASS")

Ví dụ đề + base case PASS


## 8. Các lời giải khác

### 8.1. Gộp chẵn/lẻ bằng `k % 2` (mẹo số học)

Nhận xét: phần cộng thêm `nums[k//2 + 1]` chỉ có khi `k` lẻ, tức khi `k % 2 == 1`. Nhân với `k % 2` để "bật/tắt":

```python
nums[k] = nums[k // 2] + (k % 2) * nums[k // 2 + 1]
```

Cẩn thận: với `k = n` chẵn, `k // 2 + 1` vẫn `<= n` nên không lỗi chỉ số (vì `k >= 2` ⇒ `k//2 + 1 <= k`).

In [13]:
def get_max_compact(n: int) -> int:
    if n < 2:
        return n
    nums = [0] * (n + 1)
    nums[1] = 1
    for k in range(2, n + 1):
        nums[k] = nums[k // 2] + (k % 2) * nums[k // 2 + 1]
    return max(nums)

print([get_max_compact(n) for n in [7, 2, 3, 0, 1]])

[3, 1, 2, 0, 1]


### 8.2. Đệ quy + ghi nhớ (`lru_cache`)

Định nghĩa `f(k)` = `nums[k]`:

- `f(0) = 0`, `f(1) = 1`
- `f(k) = f(k // 2)` nếu chẵn, `f(k // 2) + f(k // 2 + 1)` nếu lẻ.

`@lru_cache(maxsize=None)` là **decorator** ghi nhớ kết quả: gọi `f(5)` lần hai sẽ lấy lại từ bộ nhớ, không tính lại.

In [14]:
from functools import lru_cache

@lru_cache(maxsize=None)
def f(k: int) -> int:
    if k < 2:
        return k
    if k % 2 == 0:
        return f(k // 2)
    return f(k // 2) + f(k // 2 + 1)

def get_max_recursive(n: int) -> int:
    return max(f(k) for k in range(n + 1))

print([get_max_recursive(n) for n in [7, 2, 3, 0, 1]])
print("cache:", f.cache_info())

[3, 1, 2, 0, 1]
cache: CacheInfo(hits=19, misses=8, maxsize=None, currsize=8)


Decorator `@ten` đặt trên `def` nghĩa là `f = ten(f)`: bọc hàm bằng một lớp có thêm tính năng (ở đây là bộ nhớ đệm). Đệ quy ở bài này có độ sâu chỉ `O(log k)` vì mỗi lần chia đôi chỉ số.

### 8.3. Tính sẵn (precompute) một lần cho mọi n

Vì `n <= 100`, ta có thể sinh **một lần** mảng dài 101 và mảng `prefix_max[n]` = max của `nums[0..n]`. Mỗi truy vấn sau đó chỉ `O(1)`. Hữu ích khi có rất nhiều truy vấn.

In [15]:
LIMIT = 100
GEN = [0] * (LIMIT + 1)
GEN[1] = 1
for k in range(2, LIMIT + 1):
    GEN[k] = GEN[k // 2] + (k % 2) * GEN[k // 2 + 1]

PREFIX_MAX = [0] * (LIMIT + 1)
cur = 0
for k in range(LIMIT + 1):
    cur = max(cur, GEN[k])
    PREFIX_MAX[k] = cur

def get_max_precomputed(n: int) -> int:
    return PREFIX_MAX[n]

print("20 phần tử đầu:", GEN[:20])
print([get_max_precomputed(n) for n in [7, 2, 3, 0, 1, 100]])

20 phần tử đầu: [0, 1, 1, 2, 1, 3, 2, 3, 1, 4, 3, 5, 2, 5, 3, 4, 1, 5, 4, 7]
[3, 1, 2, 0, 1, 21]


### 8.4. Bảng so sánh các cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ | Nhận xét |
|---|---|---|---|---|
| Mô phỏng theo `i` | dịch thẳng đề | `O(n)` | `O(n)` | đúng nhưng nhiều điều kiện |
| Duyệt chỉ số đích `k` | chẵn/lẻ đọc `k//2` | `O(n)` | `O(n)` | **nên dùng** |
| Gộp bằng `k % 2` | bật/tắt số hạng | `O(n)` | `O(n)` | gọn, cần hiểu mẹo |
| Đệ quy + cache | `f(k)` từ `f(k//2)` | `O(n)` | `O(n)` | luyện tư duy đệ quy |
| Precompute | sinh một lần | `O(1)`/truy vấn sau `O(L)` | `O(L)` | tốt khi nhiều truy vấn |

(`L = 100` là giới hạn của đề.)

## 9. Vì sao lời giải đúng?

**Đúng công thức:** với `k >= 2`, viết `k = 2i` hoặc `k = 2i + 1` với `i = k // 2 >= 1`. Hai nhánh `if` chính là hai quy tắc của đề với `i = k // 2`. Điều kiện `2 <= k <= n` khớp với `range(2, n + 1)`.

**Đúng thứ tự:** khi tính `nums[k]`, ta đọc `nums[k // 2]` và (nếu lẻ) `nums[k // 2 + 1]`. Với `k >= 2`: `k // 2 < k`; với `k` lẻ `>= 3`: `k // 2 + 1 <= (k - 1) / 2 + 1 < k`. Nên mọi ô cần đọc đã được điền ở các vòng trước.

**Invariant:** sau vòng `k`, `nums[0..k]` đều đúng theo định nghĩa của đề.

**Base case:** `n = 0` → `[0]`, max `0`; `n = 1` → `[0, 1]`, max `1`. Cả hai bằng `n`, nên `if n < 2: return n`.

## 10. Độ phức tạp

Gọi `n` là input.

- **Time `O(n)`**: một vòng `for` `n - 1` lần, mỗi lần `O(1)`; cộng `max(nums)` `O(n)`.
- **Space `O(n)`**: mảng `nums` dài `n + 1`.

Có thể bớt lời gọi `max` riêng bằng cách cập nhật `best` trong vòng lặp (như hàm trace), nhưng độ phức tạp không đổi. Không thể giảm bộ nhớ xuống `O(1)` một cách đơn giản vì `nums[k]` đọc lại phần tử ở xa phía trước (`k // 2`).

## 11. Bộ test

1. 3 ví dụ đề: `7 → 3`, `2 → 1`, `3 → 2`.
2. Base case: `0 → 0`, `1 → 1`.
3. Biên trên `n = 100`.
4. Đối chiếu 5 cách với mọi `n` từ `0` đến `100` (miền input đầy đủ).
5. So mảng sinh ra với mảng ví dụ của đề.

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS -> {actual}")

check("n = 7", get_max_generated(7), 3)
check("n = 2", get_max_generated(2), 1)
check("n = 3", get_max_generated(3), 2)
check("n = 0", get_max_generated(0), 0)
check("n = 1", get_max_generated(1), 1)
check("mảng n = 7", GEN[:8], [0, 1, 1, 2, 1, 3, 2, 3])
check("n = 100", get_max_generated(100), PREFIX_MAX[100])

n = 7              PASS -> 3
n = 2              PASS -> 1
n = 3              PASS -> 2
n = 0              PASS -> 0
n = 1              PASS -> 1
mảng n = 7         PASS -> [0, 1, 1, 2, 1, 3, 2, 3]
n = 100            PASS -> 21


In [17]:
methods = [get_max_by_i, get_max_compact, get_max_recursive, get_max_precomputed,
           Solution().getMaximumGenerated]
for n in range(0, 101):
    ref = get_max_generated(n)
    for m in methods:
        assert m(n) == ref, (getattr(m, "__name__", m), n)
print("Tất cả 6 cách khớp nhau với mọi n trong [0, 100]")
print("Một vài đáp án:", {n: get_max_generated(n) for n in [10, 20, 50, 100]})

Tất cả 6 cách khớp nhau với mọi n trong [0, 100]
Một vài đáp án: {10: 4, 20: 7, 50: 13, 100: 21}


## 12. Lỗi người mới hay gặp

### Lỗi 1: Quên base case `n = 0`

Đã thấy ở mục 3: `nums[1] = 1` với mảng dài 1 gây `IndexError`. Sửa: `if n < 2: return n` trước khi tạo mảng (hoặc tạo mảng dài `max(2, n + 1)` rồi chỉ lấy max của `nums[:n + 1]`).

### Lỗi 2: Mảng dài `n` thay vì `n + 1`

Chỉ số cần tới `n`, nên mảng phải dài `n + 1`.

In [18]:
def wrong_length(n):
    nums = [0] * n
    nums[1] = 1
    for k in range(2, n + 1):
        nums[k] = nums[k // 2] + (k % 2) * nums[k // 2 + 1]
    return max(nums)

try:
    wrong_length(7)
except IndexError as e:
    print("IndexError:", e)

IndexError: list assignment index out of range


### Lỗi 3: Trả về `nums[n]` thay vì `max(nums)`

Đề hỏi **max của cả mảng**, không phải phần tử cuối. Với `n = 6`, `nums[6] = 2` nhưng max là `3`.

In [19]:
def wrong_last(n):
    if n < 2:
        return n
    return GEN[n]

print("sai :", wrong_last(6))
print("đúng:", get_max_generated(6))

sai : 2
đúng: 3


### Lỗi 4: Dùng `/` thay cho `//`

`k / 2` cho số thực (`2.5`), không dùng làm chỉ số list được.

In [20]:
nums = [0, 1, 1, 2]
try:
    print(nums[5 / 2])
except TypeError as e:
    print("TypeError:", e)
print("đúng:", nums[5 // 2])

TypeError: list indices must be integers or slices, not float
đúng: 1


### Lỗi 5: Nhầm lẻ thành `nums[k // 2] + nums[k // 2 - 1]`

Quy tắc là `nums[i] + nums[i + 1]` (cộng phần tử **sau**), không phải phần tử trước.

In [21]:
def wrong_minus(n):
    if n < 2:
        return n
    nums = [0] * (n + 1)
    nums[1] = 1
    for k in range(2, n + 1):
        nums[k] = nums[k // 2] + (k % 2) * nums[k // 2 - 1]
    return nums

print("sai :", wrong_minus(7))
print("đúng:", GEN[:8])

sai : [0, 1, 1, 1, 1, 2, 1, 2]
đúng: [0, 1, 1, 2, 1, 3, 2, 3]


## 13. Mẹo nhận dạng pattern

- Đề cho **công thức sinh** `a[2i]`, `a[2i+1]` → viết theo chỉ số đích `k` với `k // 2` và `k % 2`.
- Ô hiện tại chỉ phụ thuộc ô có chỉ số nhỏ hơn → **điền từ trái sang phải** (DP 1 chiều).
- Constraints rất nhỏ (`n <= 100`) → mô phỏng thẳng, hoặc precompute.
- Hint "base case" → luôn thử `n = 0`, `n = 1` trước khi submit.

Dãy trong bài là **dãy diatomic của Stern** (Stern's diatomic sequence), cũng xuất hiện trong cây Calkin–Wilf. Các bài cùng kiểu: 338 Counting Bits (`bits[k] = bits[k >> 1] + (k & 1)`), 509 Fibonacci Number, 1137 N-th Tribonacci Number.

## 14. Biến thể

### Biến thể 1: Counting Bits (LeetCode 338) — cùng khuôn `k // 2`, `k % 2`

Số bit 1 của `k` = số bit 1 của `k // 2` + bit cuối `k % 2`.

In [22]:
def counting_bits(n: int) -> list[int]:
    bits = [0] * (n + 1)
    for k in range(1, n + 1):
        bits[k] = bits[k // 2] + k % 2
    return bits

print(counting_bits(8))
print([bin(k).count("1") for k in range(9)], "(đối chiếu bằng bin)")

[0, 1, 1, 2, 1, 2, 2, 3, 1]
[0, 1, 1, 2, 1, 2, 2, 3, 1] (đối chiếu bằng bin)


### Biến thể 2: Vị trí đầu tiên đạt max

In [23]:
def first_index_of_max(n: int) -> int:
    nums = GEN[:n + 1]
    return nums.index(max(nums))

print([first_index_of_max(n) for n in [1, 3, 5, 7, 10]])

[1, 3, 5, 5, 9]


`list.index(v)` trả về chỉ số **đầu tiên** có giá trị `v`.

### Biến thể 3: Tổng các phần tử thay vì max

In [24]:
def sum_generated(n: int) -> int:
    return sum(GEN[:n + 1])

print([sum_generated(n) for n in [0, 1, 2, 3, 7]])

[0, 1, 2, 4, 13]


## 15. Bài tập tự luyện

Các cell TODO chỉ định nghĩa hàm, không được gọi khi chạy notebook.

### Bài 1 — Tính tay

Điền tay `nums` với `n = 12` và tìm max. (Gợi ý: tiếp từ bảng `n = 7` ở mục 1.)

### Bài 2 — Sinh mảng (không lấy max)

In [25]:
def exercise_generate(n: int) -> list[int]:
    # TODO: trả về cả mảng nums dài n + 1; chú ý n = 0
    pass

# assert exercise_generate(7) == [0, 1, 1, 2, 1, 3, 2, 3]
# assert exercise_generate(0) == [0]

### Bài 3 — Viết lại lời giải chính, cập nhật max trong vòng lặp

Không gọi `max(nums)` ở cuối; dùng biến `best`.

In [26]:
def exercise_get_max(n: int) -> int:
    # TODO 1: base case
    # TODO 2: điền nums theo chẵn/lẻ, cập nhật best
    pass

# assert exercise_get_max(7) == 3
# assert exercise_get_max(0) == 0
# assert exercise_get_max(1) == 1

### Bài 4 — Phiên bản đệ quy tự viết cache bằng dict

Không dùng `lru_cache`; tự tạo `memo = {}`.

In [27]:
def exercise_recursive(n: int) -> int:
    # TODO: định nghĩa hàm con g(k) dùng memo, trả về max(g(k) for k in 0..n)
    pass

# assert exercise_recursive(7) == 3

### Bài 5 — Biến thể: dãy với quy tắc khác

`a[0] = 1`, `a[1] = 2`; `a[2i] = a[i] + 1`, `a[2i+1] = a[i] * a[i+1]` (với `2i`, `2i+1 <= n`). Trả về max.

In [28]:
def exercise_custom(n: int) -> int:
    # TODO: đổi công thức nhưng giữ khung duyệt k
    pass

# assert exercise_custom(0) == 1
# assert exercise_custom(5) == 18

### Bài 6 — Mini challenge: Trả về mọi n trong [0, 100] mà max "tăng kỷ lục"

Một `n` là kỷ lục nếu `getMaximumGenerated(n) > getMaximumGenerated(n - 1)` (với `n >= 1`). Viết hàm trả về list các `n` kỷ lục, chỉ sinh mảng **một lần**.

In [29]:
def exercise_records(limit: int) -> list[int]:
    # TODO: sinh mảng một lần, theo dõi max chạy
    pass

# assert exercise_records(10) == [1, 3, 5, 9]

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: `k` chẵn đọc `k//2`, lẻ đọc `k//2` và `k//2+1`.
- Mức 2: `nums[8]=nums[4]=1`, `nums[9]=nums[4]+nums[5]=4`.
- Mức 3: `nums[10]=3`, `nums[11]=nums[5]+nums[6]=5`, `nums[12]=nums[6]=2`.

**Bài 2**
- Mức 1: giống lời giải chính nhưng trả về `nums`.
- Mức 2: `n = 0` trả `[0]`.
- Mức 3: `nums = [0] * (n + 1)`; `if n >= 1: nums[1] = 1`.

**Bài 3**
- Mức 1: `best` bắt đầu bằng `1` (vì `n >= 2`).
- Mức 2: sau mỗi `nums[k] = ...` gọi `best = max(best, nums[k])`.
- Mức 3: trả `best`.

**Bài 4**
- Mức 1: `memo = {0: 0, 1: 1}`.
- Mức 2: `if k in memo: return memo[k]`.
- Mức 3: tính theo chẵn/lẻ, lưu `memo[k] = value` trước khi trả.

**Bài 5**
- Mức 1: chỉ thay hai dòng công thức.
- Mức 2: base case `n = 0` → `[1]`.
- Mức 3: với `n = 5`: `a = [1, 2, 3, 6, 4, 18]`.

**Bài 6**
- Mức 1: max chạy `cur`.
- Mức 2: khi `nums[k] > cur` thì `k` là kỷ lục.
- Mức 3: `n = 1` luôn là kỷ lục (max từ 0 lên 1).

## 17. Đáp án tham khảo

**Bài 1:** `n = 12`: `[0,1,1,2,1,3,2,3,1,4,3,5,2]`, max = **5** (tại chỉ số 11).

In [30]:
def answer_generate(n):
    nums = [0] * (n + 1)
    if n >= 1:
        nums[1] = 1
    for k in range(2, n + 1):
        if k % 2 == 0:
            nums[k] = nums[k // 2]
        else:
            nums[k] = nums[k // 2] + nums[k // 2 + 1]
    return nums

def answer_get_max(n):
    if n < 2:
        return n
    nums = [0] * (n + 1)
    nums[1] = 1
    best = 1
    for k in range(2, n + 1):
        nums[k] = nums[k // 2] + (k % 2) * nums[k // 2 + 1]
        best = max(best, nums[k])
    return best

def answer_recursive(n):
    memo = {0: 0, 1: 1}
    def g(k):
        if k in memo:
            return memo[k]
        value = g(k // 2) if k % 2 == 0 else g(k // 2) + g(k // 2 + 1)
        memo[k] = value
        return value
    return max(g(k) for k in range(n + 1))

def answer_custom(n):
    a = [0] * (n + 1)
    a[0] = 1
    if n >= 1:
        a[1] = 2
    for k in range(2, n + 1):
        h = k // 2
        a[k] = a[h] + 1 if k % 2 == 0 else a[h] * a[h + 1]
    return max(a)

def answer_records(limit):
    nums = answer_generate(limit)
    cur, res = nums[0], []
    for k in range(1, limit + 1):
        if nums[k] > cur:
            res.append(k)
            cur = nums[k]
    return res

assert answer_generate(12) == [0, 1, 1, 2, 1, 3, 2, 3, 1, 4, 3, 5, 2]   # Bài 1
assert max(answer_generate(12)) == 5
assert answer_generate(7) == [0, 1, 1, 2, 1, 3, 2, 3]
assert answer_generate(0) == [0]
assert answer_get_max(7) == 3 and answer_get_max(0) == 0 and answer_get_max(1) == 1
assert answer_recursive(7) == 3
assert answer_custom(0) == 1
assert answer_custom(5) == 18
assert answer_records(10) == [1, 3, 5, 9]
for n in range(101):
    assert answer_get_max(n) == answer_recursive(n) == get_max_generated(n)
print("Tất cả đáp án PASS")
print("Các n kỷ lục tới 100:", answer_records(100))

Tất cả đáp án PASS
Các n kỷ lục tới 100: [1, 3, 5, 9, 11, 19, 21, 35, 37, 43, 69, 73, 75, 83, 85]


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def getMaximumGenerated(self, n: int) -> int:
        if n < 2:                       # 1) base case 0 và 1
            return n
        nums = [0] * (n + 1)            # 2) mảng dài n + 1
        nums[1] = 1
        for k in range(2, n + 1):       # 3) điền trái -> phải
            if k % 2 == 0:
                nums[k] = nums[k // 2]
            else:
                nums[k] = nums[k // 2] + nums[k // 2 + 1]
        return max(nums)                # 4) max cả mảng
```

Câu thần chú: **"Chỉ số đích k: chẵn đọc k//2, lẻ cộng thêm k//2 + 1."**

Checklist trước khi submit:

- [ ] Xử lý `n = 0` và `n = 1` trước khi gán `nums[1]`?
- [ ] Mảng dài `n + 1`?
- [ ] Dùng `//` (không phải `/`) cho chỉ số?
- [ ] Nhánh lẻ cộng `nums[k // 2 + 1]` (phần tử sau), không phải `- 1`?
- [ ] Trả `max(nums)`, không phải `nums[n]`?
- [ ] Đã thử `n = 0, 1, 2, 3, 7`?